Here we are going to prove that properties help in the classifiers

1. For AMP we had build ESM only classifier and now we are going to build ESM + Properties classifier

In [2]:
import numpy as np
import pandas as pd

embeddings = np.load(
    "../embeddings/amp_nonamp_embeddings.npy"
)

properties = pd.read_csv(
    "../properties/amp_nonamp_properties.csv"
)

print(embeddings.shape)
print(properties.shape)

(16800, 1280)
(16800, 7)


In [5]:
#Normalising features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

properties_scaled = scaler.fit_transform(
    properties
)

print(properties_scaled.shape)

(16800, 7)


In [6]:
import numpy as np

X_combined = np.concatenate(
    [embeddings, properties_scaled],
    axis=1
)

print(X_combined.shape)

(16800, 1287)


In [7]:
meta = pd.read_csv(
    "../embeddings/amp_nonamp_metadata.csv"
)

y = meta["label"].values

print(y.shape)

(16800,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(13440, 1287)
(3360, 1287)


In [12]:
print(properties_scaled.shape)
print(X_combined.shape)
print(X_train.shape)
print(X_test.shape)

(16800, 7)
(16800, 1287)
(13440, 1287)
(3360, 1287)


In [18]:
#Training ESM + Properties XGBoost
from xgboost import XGBClassifier

xgb_prop = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

xgb_prop.fit(
    X_train,
    y_train
)

print("Training Complete")

Training Complete


In [19]:
#Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred = xgb_prop.predict(X_test)

y_prob = xgb_prop.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print(confusion_matrix(y_test, y_pred))

Accuracy : 0.9238095238095239
Precision: 0.9253285543608124
Recall   : 0.9220238095238096
F1 Score : 0.9236732259988074
ROC-AUC  : 0.9807759353741496
[[1555  125]
 [ 131 1549]]


| Model            | Accuracy |     F1 | ROC-AUC |
| ---------------- | -------: | -----: | ------: |
| ESM Only         |   92.20% | 92.16% | 0.97996 |
| ESM + Properties |   92.38% | 92.37% | 0.98078 |


ESM embeddings already capture most physicochemical information.

2. Now we are going to build Hemolysis + Properties classifier

In [20]:
hemo_embeddings = np.load(
    "../embeddings/hemo_embeddings.npy"
)

hemo_properties = pd.read_csv(
    "../properties/hemo_properties.csv"
)

hemo_meta = pd.read_csv(
    "../embeddings/hemo_metadata.csv"
)

print(hemo_embeddings.shape)
print(hemo_properties.shape)
print(hemo_meta.shape)

(1509, 1280)
(1509, 7)
(1509, 8)


In [21]:
#Creating feature matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

hemo_props_scaled = scaler.fit_transform(
    hemo_properties
)

X_hemo_combined = np.concatenate(
    [hemo_embeddings, hemo_props_scaled],
    axis=1
)

print(X_hemo_combined.shape)

(1509, 1287)


In [22]:
y_hemo = hemo_meta["hemo_label"].values

print(y_hemo.shape)

(1509,)


In [23]:
#Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_hemo_combined,
    y_hemo,
    test_size=0.2,
    random_state=42,
    stratify=y_hemo
)

print(X_train.shape)
print(X_test.shape)

(1207, 1287)
(302, 1287)


In [24]:
#Training XGBoost
from xgboost import XGBClassifier

hemo_xgb_prop = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

hemo_xgb_prop.fit(
    X_train,
    y_train
)

print("Training Complete")

Training Complete


In [25]:
#Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred = hemo_xgb_prop.predict(X_test)
y_prob = hemo_xgb_prop.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7947019867549668
Precision: 0.7865168539325843
Recall   : 0.6194690265486725
F1 Score : 0.693069306930693
ROC-AUC  : 0.8753570257995037
[[170  19]
 [ 43  70]]


| Model            | Accuracy |    F1 | ROC-AUC |
| ---------------- | -------: | ----: | ------: |
| ESM Only         |   79.47% | 0.699 |   0.877 |
| ESM + Properties |   79.47% | 0.693 |   0.875 |


There is no meaningful improvement, keeping the old classifier as best classifier becoz ESM embeddings already capture
most of the useful physicochemical information.